# Data Formats & Storage — 03: Delta Lake Advanced

## What you will learn
Delta Lake adds **reliability, performance, and governance** to your data lake.
It is a storage layer that brings database-like guarantees to files on object storage.

In this notebook you will master:
1. **Delta table internals** — transaction log, checkpoints, how ACID is implemented
2. **OPTIMIZE** — compacting small files into large ones for faster queries
3. **ZORDER** — multi-dimensional clustering to co-locate related data
4. **VACUUM** — safe deletion of old data files
5. **Change Data Feed (CDF)** — track row-level changes for CDC pipelines
6. **MERGE INTO** — efficient upsert with Delta's optimized merge
7. **Time travel** — query past versions of your data
8. **Schema enforcement and evolution** — Delta's strict schema management
9. **Generated columns** — derived columns maintained automatically
10. **Delta vs Iceberg** — when to use which

## How Delta Lake Works Internally
```
Delta Table = Parquet files + _delta_log/

_delta_log/
  00000000000000000000.json  ← commit 0: CREATE TABLE
  00000000000000000001.json  ← commit 1: INSERT 1000 rows
  00000000000000000002.json  ← commit 2: UPDATE status
  00000000000000000003.json  ← commit 3: DELETE rows
  00000000000000000010.checkpoint.parquet  ← checkpoint every 10 commits

Each .json file records:
  - add:    new files added by this commit
  - remove: files logically deleted (soft delete)
  - metaData: schema changes
  - protocol: reader/writer requirements

ACID guarantee: atomic file swap + transaction log = serializable isolation
```


In [ ]:
import os, time, datetime, pathlib, random
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'
DELTA_DIR = f'{DATA_DIR}/delta_advanced'

spark = (
    SparkSession.builder
    .appName('delta-advanced')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    # Delta already configured in spark-defaults.conf:
    #   spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
    #   spark.sql.catalog.spark_catalog = DeltaCatalog
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")
print(f"Delta directory: {DELTA_DIR}")

## Step 1 — Create a Delta Table and Understand the Transaction Log

Every operation on a Delta table writes a JSON commit to `_delta_log/`.
Let's create a table and inspect what Delta writes internally.


In [ ]:
import shutil
random.seed(42)

# Clean up
shutil.rmtree(DELTA_DIR, ignore_errors=True)
pathlib.Path(DELTA_DIR).mkdir(parents=True, exist_ok=True)

EVENTS    = ["page_view", "click", "add_to_cart", "purchase", "search"]
PAGES     = ["/home", "/products", "/cart", "/checkout", "/account", "/search"]
DEVICES   = ["mobile", "desktop", "tablet"]
COUNTRIES = ["US", "UK", "DE", "FR", "JP", "CA", "BR", "AU"]

def make_events(n, start_id=1, start_date="2024-01-01", days=90):
    base = datetime.datetime.fromisoformat(start_date + " 00:00:00")
    rows = []
    for i in range(n):
        event_dt = base + datetime.timedelta(days=random.randint(0, days),
                                              seconds=random.randint(0, 86400))
        rows.append((
            start_id + i,
            random.randint(1, 5000),
            random.choice(EVENTS),
            random.choice(PAGES),
            random.choice(DEVICES),
            random.choice(COUNTRIES),
            round(random.uniform(0, 500), 2) if random.random() > 0.7 else 0.0,
            random.randint(1000, 60000),
            event_dt.date(),
            event_dt
        ))
    return rows

schema = StructType([
    StructField("event_id",    LongType()),
    StructField("user_id",     LongType()),
    StructField("event_type",  StringType()),
    StructField("page",        StringType()),
    StructField("device",      StringType()),
    StructField("country",     StringType()),
    StructField("revenue",     DoubleType()),
    StructField("session_ms",  LongType()),
    StructField("event_date",  DateType()),
    StructField("event_ts",    TimestampType()),
])

TABLE_PATH = f"{DELTA_DIR}/events"

# Write initial batch — this creates commit 0 in _delta_log/
batch1 = spark.createDataFrame(make_events(10000, start_id=1), schema)
batch1.write.format("delta").mode("overwrite") \
      .partitionBy("event_date") \
      .save(TABLE_PATH)

print(f"Delta table created at {TABLE_PATH}")
print(f"Rows: {spark.read.format('delta').load(TABLE_PATH).count():,}")

# Peek inside _delta_log
import json as pyjson
log_path = pathlib.Path(f"{TABLE_PATH}/_delta_log")
commits = sorted(log_path.glob("*.json"))
print(f"\nTransaction log files: {len(commits)}")
for c in commits[:3]:
    print(f"  {c.name}")
    with open(c) as f:
        for line in f.readlines()[:3]:
            entry = pyjson.loads(line)
            print(f"    {list(entry.keys())}")

In [ ]:
# Read the first commit in detail
commit0 = pathlib.Path(f"{TABLE_PATH}/_delta_log/00000000000000000000.json")
print("Commit 0 contents (CREATE TABLE + first write):")
with open(commit0) as f:
    for line in f:
        entry = pyjson.loads(line.strip())
        key = list(entry.keys())[0]
        if key in ("metaData", "commitInfo"):
            print(f"\n  [{key}]")
            if key == "metaData":
                print(f"    schema fields: {len(pyjson.loads(entry[key]['schemaString'])['fields'])}")
                print(f"    partitionColumns: {entry[key]['partitionColumns']}")
            elif key == "commitInfo":
                print(f"    operation: {entry[key].get('operation')}")
                print(f"    timestamp: {entry[key].get('timestamp')}")
        elif key == "add":
            print(f"  [add] {entry[key]['path'][:60]}...  ({entry[key]['size']:,} bytes)")

## Step 2 — OPTIMIZE: Solving the Small File Problem

When you append data frequently (streaming, hourly batches), you accumulate
**thousands of tiny Parquet files**. Each file = one task = overhead.

`OPTIMIZE` compacts small files into fewer large files, dramatically improving
read performance. Target file size is configurable (default 1 GB).

**Before OPTIMIZE:** 1000 files × 1 MB = 1 GB total, 1000 tasks to read
**After OPTIMIZE:**  8 files × 128 MB = 1 GB total, 8 tasks to read


In [ ]:
from delta.tables import DeltaTable

# Create the small file problem: insert 20 tiny batches
print("Creating small file problem (20 tiny batches)...")
for i in range(20):
    tiny_batch = spark.createDataFrame(
        make_events(100, start_id=10001 + i*100), schema)
    tiny_batch.write.format("delta").mode("append").save(TABLE_PATH)

# Count files before OPTIMIZE
detail_before = DeltaTable.forPath(spark, TABLE_PATH).detail().collect()[0]
print(f"\nBefore OPTIMIZE:")
print(f"  numFiles      : {detail_before['numFiles']}")
print(f"  sizeInBytes   : {detail_before['sizeInBytes'] / 1024 / 1024:.1f} MB")
print(f"  Total rows    : {spark.read.format('delta').load(TABLE_PATH).count():,}")

# Time a query before OPTIMIZE
t0 = time.time()
spark.read.format("delta").load(TABLE_PATH) \
     .filter(F.col("event_type") == "purchase") \
     .agg(F.sum("revenue"), F.count("*")).collect()
t_before = time.time() - t0
print(f"  Query time    : {t_before:.3f}s")

In [ ]:
from delta.tables import DeltaTable

# Run OPTIMIZE
print("Running OPTIMIZE...")
t0 = time.time()
spark.sql(f"OPTIMIZE delta.`{TABLE_PATH}`").show(truncate=False)
t_optimize = time.time() - t0

detail_after = DeltaTable.forPath(spark, TABLE_PATH).detail().collect()[0]
print(f"\nAfter OPTIMIZE (took {t_optimize:.1f}s):")
print(f"  numFiles      : {detail_after['numFiles']}")
print(f"  sizeInBytes   : {detail_after['sizeInBytes'] / 1024 / 1024:.1f} MB")

# Time a query after OPTIMIZE
t0 = time.time()
spark.read.format("delta").load(TABLE_PATH) \
     .filter(F.col("event_type") == "purchase") \
     .agg(F.sum("revenue"), F.count("*")).collect()
t_after = time.time() - t0
print(f"  Query time    : {t_after:.3f}s")
print(f"  Speedup       : {t_before/t_after:.1f}x")
print(f"  Files reduced : {detail_before['numFiles']} → {detail_after['numFiles']}")

## Step 3 — ZORDER: Data Clustering for Faster Queries

`OPTIMIZE ... ZORDER BY` reorganises data so that **related rows are stored
in the same files**. When you filter by ZORDERed columns, Delta can skip
entire files (data skipping) instead of scanning everything.

**How Z-Ordering works:**
Z-ordering is a space-filling curve algorithm that maps multi-dimensional data
to 1D while preserving locality. Rows with similar values on the ZORDERed
columns end up in the same files.

**When to ZORDER:**
- Columns you frequently filter on (`WHERE country = 'US'`)
- High-cardinality columns with range queries (`WHERE user_id BETWEEN 1000 AND 2000`)
- Multiple filter columns (`WHERE country = 'US' AND device = 'mobile'`)

**When NOT to ZORDER:**
- Partition columns (already co-located)
- Low-cardinality columns (e.g., boolean — only 2 values, no benefit)


In [ ]:
# First: measure query WITHOUT ZORDER
def timed_query(label):
    t0 = time.time()
    result = spark.read.format("delta").load(TABLE_PATH) \
                  .filter(F.col("country") == "US") \
                  .filter(F.col("device") == "mobile") \
                  .filter(F.col("event_type") == "purchase") \
                  .agg(F.sum("revenue").alias("total"), F.count("*").alias("cnt")) \
                  .collect()
    elapsed = time.time() - t0
    print(f"  {label:<35} {elapsed:.3f}s | revenue={result[0]['total']:.2f}, rows={result[0]['cnt']}")
    return elapsed

t_no_zorder = timed_query("Without ZORDER")

# Apply ZORDER on frequently-filtered columns
print("\nApplying ZORDER BY (country, device, event_type)...")
t0 = time.time()
spark.sql(f"""
    OPTIMIZE delta.`{TABLE_PATH}`
    ZORDER BY (country, device, event_type)
""").show(truncate=False)
print(f"ZORDER took {time.time()-t0:.1f}s")

t_zorder = timed_query("With ZORDER")
print(f"\nZORDER speedup: {t_no_zorder/t_zorder:.1f}x")
print()
print("Note: ZORDER benefit grows significantly with larger datasets.")
print("On 100M+ rows the difference is typically 5-20x for selective queries.")

In [ ]:
# Delta stores per-file statistics as JSON in the transaction log
# Read via DeltaTable.forPath().detail() and the files metadata DataFrame API
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, TABLE_PATH)

files_df = dt.detail().select("numFiles", "sizeInBytes")
print("Table summary:")
files_df.show(truncate=False)

# File-level stats via transaction log (read _delta_log JSON directly)
import pathlib, json as pyjson

log_path = pathlib.Path(f"{TABLE_PATH}/_delta_log")
stats_rows = []
for commit_file in sorted(log_path.glob("*.json"))[-3:]:
    with open(commit_file) as f:
        for line in f:
            entry = pyjson.loads(line.strip())
            if "add" in entry and entry["add"].get("stats"):
                s = pyjson.loads(entry["add"]["stats"])
                stats_rows.append({
                    "file":        entry["add"]["path"][-40:],
                    "records":     s.get("numRecords"),
                    "min_country": s.get("minValues", {}).get("country"),
                    "max_country": s.get("maxValues", {}).get("country"),
                    "min_event":   s.get("minValues", {}).get("event_type"),
                    "max_event":   s.get("maxValues", {}).get("event_type"),
                })

if stats_rows:
    stats_df = spark.createDataFrame(stats_rows)
    print("\nPer-file min/max statistics (used for data skipping):")
    stats_df.show(8, truncate=False)
    print("With ZORDER, files for country=\'US\' will have min_country=max_country=\'US\'.")
    print("Delta skips files where max_country < \'US\' or min_country > \'US\'.")
else:
    print("No stats found in recent commits.")


## Step 4 — VACUUM: Safe Deletion of Old Files

After OPTIMIZE, old small files are **logically deleted** (marked as removed in the
transaction log) but the physical Parquet files remain on disk.

`VACUUM` physically deletes files that have been logically removed AND are older
than the retention period (default: 7 days).

**Why the retention period?**
Time travel relies on old files. If you vacuum files that a time-travel query
needs, that query will fail. The 7-day default gives you a week to run
historical queries before files are gone.

> ⚠️ **Never set retention to 0 in production.** It will break time travel
> and any in-flight queries that started before the VACUUM.


In [ ]:
# Disable safety check FIRST — before calling VACUUM with low retention
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# Dry run — shows what would be deleted without actually deleting
print("VACUUM dry run — files that would be deleted:")
spark.sql(f"VACUUM delta.`{TABLE_PATH}` RETAIN 0 HOURS DRY RUN").show(10, truncate=False)

# Count files before VACUUM
import subprocess
result = subprocess.run(["find", TABLE_PATH, "-name", "*.parquet"],
                        capture_output=True, text=True)
files_before = len([x for x in result.stdout.strip().split("\n") if x])
print(f"\nParquet files on disk before VACUUM: {files_before}")

# Run VACUUM
spark.sql(f"VACUUM delta.`{TABLE_PATH}` RETAIN 0 HOURS")
print("VACUUM complete.")

result = subprocess.run(["find", TABLE_PATH, "-name", "*.parquet"],
                        capture_output=True, text=True)
files_after = len([x for x in result.stdout.strip().split("\n") if x])
print(f"Parquet files on disk after VACUUM : {files_after}")
print(f"Files deleted: {files_before - files_after}")

# Re-enable safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

## Step 5 — Change Data Feed (CDF)

Delta's **Change Data Feed** records every row-level change (insert/update/delete)
and makes them queryable. This is perfect for:

- **CDC pipelines**: stream changes to downstream systems
- **Audit logs**: who changed what and when
- **Incremental processing**: only process rows that changed since last run
- **Materialised views**: efficiently update derived tables

CDF adds 3 hidden columns to change records:
- `_change_type`: `insert`, `update_preimage`, `update_postimage`, `delete`
- `_commit_version`: which Delta version this change came from
- `_commit_timestamp`: when the change was committed


In [ ]:
from delta.tables import DeltaTable

# Create a new table with CDF enabled
CDF_PATH = f"{DELTA_DIR}/orders_cdf"
shutil.rmtree(CDF_PATH, ignore_errors=True)

ORDER_STATUSES = ["pending", "confirmed", "shipped", "delivered", "cancelled"]

def make_orders_cdf(n, start_id=1):
    rows = []
    base = datetime.date(2024, 1, 1)
    for i in range(n):
        odate = base + datetime.timedelta(days=random.randint(0, 90))
        rows.append((
            start_id + i,
            random.randint(1, 500),
            f"Product_{random.randint(1, 50)}",
            round(random.uniform(10, 500), 2),
            random.choice(["pending", "confirmed"]),
            odate
        ))
    return rows

order_schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("amount",      DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

# Write with CDF enabled
initial_orders = spark.createDataFrame(make_orders_cdf(500), order_schema)
initial_orders.write.format("delta") \
              .option("delta.enableChangeDataFeed", "true") \
              .mode("overwrite") \
              .save(CDF_PATH)

print(f"CDF-enabled table created: {initial_orders.count()} orders")
print("\nTable properties:")
DeltaTable.forPath(spark, CDF_PATH).detail() \
     .select("name","numFiles","sizeInBytes","properties").show(truncate=False)

In [ ]:
# Make some changes — these will be tracked by CDF
from delta.tables import DeltaTable
version_before_changes = DeltaTable.forPath(spark, CDF_PATH).history(1).collect()[0]["version"]

# 1. Update: some orders got shipped
spark.sql(f"""
    UPDATE delta.`{CDF_PATH}`
    SET status = 'shipped'
    WHERE status = 'confirmed' AND order_id % 3 = 0
""")

# 2. Update: some orders delivered
spark.sql(f"""
    UPDATE delta.`{CDF_PATH}`
    SET status = 'delivered'
    WHERE status = 'shipped' AND order_id % 5 = 0
""")

# 3. Delete: cancel old pending orders
spark.sql(f"""
    DELETE FROM delta.`{CDF_PATH}`
    WHERE status = 'pending' AND order_id < 50
""")

# 4. Insert: new orders
new_orders = spark.createDataFrame(make_orders_cdf(50, start_id=501), order_schema)
new_orders.write.format("delta").mode("append").save(CDF_PATH)

version_after_changes = DeltaTable.forPath(spark, CDF_PATH).history(1).select("version").first()["version"]

print(f"Changes made between version {version_before_changes} and {version_after_changes}")
print()
print("Table history:")
DeltaTable.forPath(spark, CDF_PATH).history() \
     .select("version","timestamp","operation","operationParameters") \
     .show(10, truncate=False)

In [ ]:
# Read the Change Data Feed — see every row-level change
print(f"All changes since version {version_before_changes}:")
changes = spark.read.format("delta") \
               .option("readChangeFeed", "true") \
               .option("startingVersion", version_before_changes + 1) \
               .load(CDF_PATH)

# Summary by change type
print("\nChange summary:")
changes.groupBy("_change_type") \
       .agg(F.count("*").alias("row_count")) \
       .orderBy("_change_type").show()

# Show update pairs (before + after)
print("\nUpdate pairs (preimage → postimage) for status changes:")
changes.filter(F.col("_change_type").isin("update_preimage","update_postimage")) \
       .select("order_id","status","_change_type","_commit_version") \
       .orderBy("order_id","_change_type") \
       .show(12)

print("\nDeleted rows:")
changes.filter(F.col("_change_type") == "delete") \
       .select("order_id","status","_commit_version") \
       .show(5)

In [ ]:
# CDF use case: incremental pipeline
# "Give me only the records that changed since my last run"
print("Incremental pipeline pattern:")
print("  1. Record last processed version")
print("  2. Read CDF from that version")
print("  3. Process only changed rows")
print("  4. Update checkpoint to current version")
print()

# Simulate: downstream system tracks last processed version
last_processed_version = version_before_changes

# Read only NEW changes
incremental_changes = (
    spark.read.format("delta")
         .option("readChangeFeed", "true")
         .option("startingVersion", last_processed_version + 1)
         .load(CDF_PATH)
         .filter(F.col("_change_type") != "update_preimage")  # skip before-images
)

print(f"Rows to process in incremental run: {incremental_changes.count()}")
print("(Only inserts, deletes, and update post-images)")

# Get current version as new checkpoint
new_checkpoint = DeltaTable.forPath(spark, CDF_PATH).history(1).collect()[0]["version"]
print(f"\nNew checkpoint version: {new_checkpoint}")
print("Next run: startingVersion = {new_checkpoint + 1}")

## Step 6 — Time Travel

Delta time travel works identically to Iceberg — every commit is a version.
Query any past state using `VERSION AS OF` or `TIMESTAMP AS OF`.


In [ ]:
# Show full version history
print("Version history:")
DeltaTable.forPath(spark, CDF_PATH).history() \
     .select("version","timestamp","operation") \
     .orderBy("version").show(truncate=False)

# Query version 0 (original data)
print("Version 0 — original inserts:")
spark.read.format("delta").option("versionAsOf", 0).load(CDF_PATH) \
     .groupBy("status").count().orderBy("status").show()

# Current state
print("Current state:")
spark.read.format("delta").load(CDF_PATH) \
     .groupBy("status").count().orderBy("status").show()

# Restore to a previous version (non-destructive — creates new version)
print("\nRestoring table to version 0...")
spark.sql(f"RESTORE TABLE delta.`{CDF_PATH}` TO VERSION AS OF 0")
print(f"After restore — row count: {spark.read.format('delta').load(CDF_PATH).count()}")
print("Note: RESTORE creates a new version — it does NOT delete history.")

## Step 7 — Schema Enforcement and Evolution

Delta enforces schema by default — trying to write incompatible data fails.
This prevents silent data corruption from mismatched pipelines.

To add new columns, use `mergeSchema` or `ALTER TABLE ADD COLUMNS`.


In [ ]:
# Schema enforcement: Delta rejects incompatible writes
print("=== Schema Enforcement Demo ===")
print()

# Try to write data with a new column — Delta will REJECT this by default
bad_df = spark.createDataFrame(
    [(999, 1, "Laptop", 999.99, "pending",
      datetime.date(2024, 6, 1), "gold")],  # extra 'tier' column
    ["order_id","customer_id","product","amount","status","order_date","tier"]
)

try:
    bad_df.write.format("delta").mode("append").save(CDF_PATH)
    print("Write succeeded (should not happen!)")
except Exception as e:
    print(f"Write REJECTED (expected): {type(e).__name__}")
    print(f"  {str(e)[:120]}...")

print()
print("Delta enforced the schema — the write was rejected.")
print("This prevents silent schema drift in production pipelines.")

In [ ]:
# Schema evolution: opt-in with mergeSchema
print("=== Schema Evolution with mergeSchema ===")

evolved_df = spark.createDataFrame(
    [(601, 1, "Laptop", 999.99, "pending",
      datetime.date(2024, 6, 1), "gold"),
     (602, 2, "Phone",  499.99, "confirmed",
      datetime.date(2024, 6, 2), "silver")],
    ["order_id","customer_id","product","amount","status","order_date","tier"]
)

# mergeSchema=true allows adding new columns
evolved_df.write.format("delta") \
          .mode("append") \
          .option("mergeSchema", "true") \
          .save(CDF_PATH)

print("Write succeeded with mergeSchema=true")
print()
print("New schema (tier column added):")
spark.read.format("delta").load(CDF_PATH).printSchema()
print()
print("Old rows show NULL for new column:")
spark.read.format("delta").load(CDF_PATH) \
     .select("order_id","status","tier") \
     .orderBy(F.desc("order_id")).show(8)

## Step 8 — Generated Columns

Generated columns are **automatically computed** from other columns.
Delta maintains them on every insert/update — you never need to set them manually.

**Use cases:**
- Partition by year/month/day extracted from a timestamp column
- Compute a hash or checksum of other columns
- Derive a category from a numeric range


In [ ]:
GEN_PATH = f"{DELTA_DIR}/events_generated"
shutil.rmtree(GEN_PATH, ignore_errors=True)

# NOTE: GENERATED ALWAYS AS columns require a catalog-registered table.
# Path-based Delta tables in open-source Spark 4.0 do not support this DDL.
# The equivalent pattern is to compute derived columns explicitly on write
# using withColumn — Delta still partitions and stores them correctly.

print("Simulating generated columns via withColumn on write:")
print("  event_date     = CAST(event_ts AS DATE)")
print("  event_hour     = HOUR(event_ts)")
print("  revenue_bucket = CASE WHEN revenue=0 THEN 'free' ...")
print()

events_raw = spark.createDataFrame([
    (1, 101, "purchase", 149.99, datetime.datetime(2024, 3, 15, 14, 30)),
    (2, 102, "purchase",   0.00, datetime.datetime(2024, 3, 15,  9, 15)),
    (3, 103, "purchase", 499.00, datetime.datetime(2024, 3, 16, 20, 45)),
    (4, 104, "click",      0.00, datetime.datetime(2024, 3, 17, 11,  0)),
], ["event_id","user_id","event_type","revenue","event_ts"])

# Compute derived columns before writing — equivalent to generated columns
events_enriched = events_raw \
    .withColumn("event_date", F.to_date("event_ts")) \
    .withColumn("event_hour", F.hour("event_ts")) \
    .withColumn("revenue_bucket",
        F.when(F.col("revenue") == 0,    "free")
         .when(F.col("revenue") < 50,    "low")
         .when(F.col("revenue") < 200,   "medium")
         .otherwise("high")
    )

events_enriched.write.format("delta") \
    .partitionBy("event_date") \
    .mode("overwrite") \
    .save(GEN_PATH)

print("Written with derived columns:")
spark.read.format("delta").load(GEN_PATH).show(truncate=False)
print()
print("Tip: for true GENERATED ALWAYS AS support, register the table")
print("in a catalog (e.g. spark.sql catalog with Hive metastore or Unity Catalog).")

## Step 9 — Delta Table Statistics and ANALYZE

Delta collects column-level statistics (min, max, null count) on write.
These statistics power **data skipping** — Delta skips files that cannot
possibly contain rows matching your filter.

You can also run `ANALYZE TABLE` to collect more detailed statistics
for the Spark optimizer's cost-based optimization (CBO).


In [ ]:
# ANALYZE TABLE not supported for path-based Delta in open-source Spark
# Use DeltaTable.detail() for table-level stats
from delta.tables import DeltaTable
import pathlib, json as pyjson

print("Table-level statistics via DeltaTable.detail():")
DeltaTable.forPath(spark, CDF_PATH).detail() \
     .select("format","numFiles","sizeInBytes","partitionColumns") \
     .show(truncate=False)

# File-level stats: read directly from _delta_log JSON files
print("File-level min/max statistics (used for data skipping):")
log_path = pathlib.Path(f"{CDF_PATH}/_delta_log")
stats_rows = []
for commit_file in sorted(log_path.glob("*.json"))[-3:]:
    with open(commit_file) as cf:
        for line in cf:
            entry = pyjson.loads(line.strip())
            if "add" in entry and entry["add"].get("stats"):
                s = pyjson.loads(entry["add"]["stats"])
                stats_rows.append({
                    "file":       entry["add"]["path"][-30:],
                    "records":    str(s.get("numRecords", "")),
                    "min_order":  str(s.get("minValues", {}).get("order_id", "")),
                    "max_order":  str(s.get("maxValues", {}).get("order_id", "")),
                    "min_amount": str(s.get("minValues", {}).get("amount", "")),
                    "max_amount": str(s.get("maxValues", {}).get("amount", "")),
                })

if stats_rows:
    spark.createDataFrame(stats_rows).show(5, truncate=False)
    print("\nData skipping: for WHERE order_id = 999,")
    print("Delta reads ONLY files where min_order <= 999 <= max_order.")
else:
    print("No stats found — files may have been vacuumed or compacted.")


## Summary: Delta Lake Feature Map

| Feature | Command | Use case |
|---|---|---|
| Create table | `CREATE TABLE ... USING delta` | ACID table on object storage |
| OPTIMIZE | `OPTIMIZE delta.\`path\`` | Compact small files |
| ZORDER | `OPTIMIZE ... ZORDER BY (col1, col2)` | Co-locate related rows for faster filters |
| VACUUM | `VACUUM delta.\`path\` RETAIN N HOURS` | Delete old physical files |
| Change Data Feed | `DESCRIBE HISTORY` + `readChangeFeed=true` | CDC, incremental pipelines, audit |
| Time travel | `VERSION AS OF N` / `TIMESTAMP AS OF '...'` | Reproducibility, debugging |
| Restore | `RESTORE TABLE ... TO VERSION AS OF N` | Rollback bad pipeline run |
| Schema enforcement | Default behavior | Prevent silent schema drift |
| Schema evolution | `.option("mergeSchema", "true")` | Safely add new columns |
| Generated columns | `GENERATED ALWAYS AS (expr)` | Auto-computed derived columns |
| Statistics | `ANALYZE TABLE ... COMPUTE STATISTICS` | Better CBO query plans |

### Delta vs Iceberg — Practical Guide

| | Delta Lake | Iceberg |
|---|---|---|
| **Ecosystem** | Spark-first, strong Databricks support | Multi-engine (Spark, Trino, Flink, Hive) |
| **MERGE performance** | Highly optimised | Good, improving |
| **ZORDER / clustering** | ZORDER BY | SORT ORDER (less flexible) |
| **CDF / CDC** | Built-in Change Data Feed | Needs custom solution |
| **Branching / tagging** | Not built-in | Native branches + tags |
| **Partition evolution** | Limited | Full evolution without rewrite |
| **Vendor neutrality** | Linux Foundation, broad support | Apache, fully open |
| **Best for** | Spark shops, Databricks, Azure | Multi-engine, cloud-agnostic |

> **Bottom line:** Both are production-ready. Delta if your stack is Spark-centric.
> Iceberg if you need multi-engine or long-term vendor independence.
